# CLO Cashflow Forecast Workflow

This notebook walks through the complete CLO cashflow forecasting process,
replacing the VBA macro workbook (`Forecast v16.3.xlsm`) with Python.

## Purpose

Model how CLO portfolio **cashflows and balances evolve** under different AAA spread scenarios.
Tighter spreads mean more deals get called (refinanced) sooner, returning principal faster.
Wider spreads mean fewer calls, extending the portfolio's weighted average life.

## Workflow Overview

| Step | What | Where | Output |
|------|------|-------|--------|
| 1 | Import CLO holdings | Python | Holdings DataFrame |
| 2 | Enrich with Intex/BBG data, clean | Excel then Python | Cleaned holdings + Tranches |
| 3 | Clear forecasted factors | Python | Empty factors table |
| 4 | Generate preliminary Intex upload (COLLAT) | Python | `PreliminaryPortfolioUpload.xlsx` |
| 4b | Run in IntexCalc | **Manual** | Preliminary cashflows export |
| 5 | Extract factor call dates from COLLAT cashflows | Python | Factor call dates table |
| 6 | Generate final Intex upload (actual tranches) | Python | `FinalPortfolioUpload.xlsx` |
| 6b | Run in IntexCalc | **Manual** | Final cashflows export |
| 7 | Summarize cashflows, allocate to sub-portfolios | Python | `CashflowSummary.xlsx`, `AllocatedCashflows.xlsx` |

---
## Setup

In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# Add structured_products library to PYTHONPATH (update path for your environment)
sys.path.append('C:/ActData/Python/files/OOI/structured-products-main/')

# Forecast modules
from forecast.config import CONFIG, ScenarioDefinition
from forecast.factors import ForecastedFactors
from forecast.forward_curve import load_from_workbook, load_from_cashflows_report
from forecast.holdings import (
    clean_holdings,
    enrich_holdings,
    export_tranches,
    import_holdings,
    load_holdings_from_clo_library,
    load_holdings_from_forecast_workbook,
)
from forecast.bloomberg import enrich_with_bbg
from forecast.preprice import PrePriceDeals
from forecast.scenarios import (
    generate_portfolio_upload,
    load_scenarios_from_config,
    load_scenarios_from_workbook,
    write_portfolio_upload,
)
from forecast.cashflows_summary import generate_cashflow_summary, load_cashflows_reports
from forecast.portfolio import (
    SubPortfolio,
    compute_weights,
    allocate_cashflows,
    one_per_entity,
    one_per_pam,
)

---
## Configuration

Set your input file paths and output directory here.

In [ ]:
# ── INPUT FILES ──────────────────────────────────────────────────────────

# The Forecast workbook — contains pre-price deals, forecasted factors,
# and forward curve.  Also used as fallback for holdings if the clo library
# is not available.
FORECAST_WB = "Forecast v16.3.xlsm"

# Holdings source (pick ONE):
#   "clo_library"   — Default. Pulls live holdings via structured_products.clo.holdings()
#   "data_packet"   — Import from a Structured Products Data Packet .xlsx
#   "forecast_wb"   — Load from the Forecast workbook's Initial Holdings sheet
HOLDINGS_SOURCE = "clo_library"

# Only used if HOLDINGS_SOURCE = "data_packet":
DATA_PACKET = None  # e.g. "Structured Products Data Packet.xlsx"

# Preliminary Intex Cashflows export (output of Step 4b).
# Set to None if you haven't run IntexCalc yet (will load from workbook).
PRELIMINARY_CASHFLOWS = None  # e.g. "PreliminaryCashflows.xlsx"

# Final Intex Cashflows export (output of Step 6b).
# Set to None to skip Step 7.
FINAL_CASHFLOWS = None  # e.g. "FinalCashflows.xlsx"

# ── OUTPUT ────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
TODAY = date.today().strftime("%Y%m%d")

---
## Scenario Configuration

This is where you set **where you think AAA CLO spreads are today** and
**what spread shocks you want to model**.

The call decision for each deal in each scenario is:

```
strike = current_aaa_margin + shock + refi_costs (10 bps)
                                    + mm_basis   (50 bps, if middle market)
                                    - libor_adj  (26 bps, if LIBOR-based)

if deal_aaa_margin > strike  →  deal gets called at non-call end date
```

**Tighter scenarios** (negative shock) → lower strike → more deals called → faster principal return.

**Wider scenarios** (positive shock) → higher strike → fewer deals called → longer WAL.

In [ ]:
# ── CURRENT MARKET SPREAD ────────────────────────────────────────────────

CONFIG.scenario.current_aaa_margin_bps = 115  # Where BSL AAA new-issue spreads are today (bps)
CONFIG.scenario.settle_date = date(2026, 2, 18)  # Settlement date for all scenarios
CONFIG.scenario.prepay_speed = 15  # CPR assumption

# ── SPREAD SCENARIOS ─────────────────────────────────────────────────────
# Each scenario applies a shock to the current spread.
# Negative = tightening, positive = widening.

CONFIG.scenario.scenarios = [
    ScenarioDefinition("75 bps Tightening", -75),
    ScenarioDefinition("50 bps Tightening", -50),
    ScenarioDefinition("25 bps Tightening", -25),
    ScenarioDefinition("10 bps Tightening", -10),
    ScenarioDefinition("Flat", 0),
    ScenarioDefinition("10 bps Widening", 10),
    ScenarioDefinition("25 bps Widening", 25),
    ScenarioDefinition("50 bps Widening", 50),
]

# ── OTHER ASSUMPTIONS (change if needed) ─────────────────────────────────

# CONFIG.call.refi_costs_bps = 10                    # Refi friction
# CONFIG.call.middle_market_bsl_basis_bps = 50       # MM AAA spread premium
# CONFIG.call.libor_sofr_basis_bps = 26.161          # LIBOR-SOFR adjustment
# CONFIG.defaults.default_rate = 5                   # CDR (%)
# CONFIG.defaults.severity_pct = 50                  # Loss severity (%)
# CONFIG.defaults.recovery_lag_months = 12           # Recovery lag (months)
# CONFIG.factor_call.factor_threshold = 0.35         # Deal factor call trigger

print("Current AAA margin:", CONFIG.scenario.current_aaa_margin_bps, "bps")
print("Settle date:", CONFIG.scenario.settle_date)
print(f"Running {len(CONFIG.scenario.scenarios)} scenarios:")
for defn in CONFIG.scenario.scenarios:
    strike = CONFIG.scenario.current_aaa_margin_bps + defn.aaa_margin_shock_bps + CONFIG.call.refi_costs_bps
    print(f"  {defn.name:25s}  shock={defn.aaa_margin_shock_bps:+.0f} bps  →  BSL call strike={strike:.0f} bps")

---
## Step 1: Import CLO Holdings

Load all CLO positions from one of three sources:

1. **`clo_library`** (default) — Pulls live holdings via `structured_products.clo.holdings()`.
   This is the same method used in the `holdings_pricing` notebook.
   Requires the `structured_products` package on your PYTHONPATH.

2. **`data_packet`** — Import from a Structured Products Data Packet `.xlsx` file.

3. **`forecast_wb`** — Load from the Forecast workbook's Initial Holdings sheet
   (useful when Intex/BBG formulas are already populated there).

In [ ]:
if HOLDINGS_SOURCE == "data_packet" and DATA_PACKET:
    holdings_df = import_holdings(DATA_PACKET)
elif HOLDINGS_SOURCE == "forecast_wb":
    holdings_df = load_holdings_from_forecast_workbook(FORECAST_WB)
else:
    # Default: load from structured_products.clo library
    try:
        holdings_df = load_holdings_from_clo_library()
    except ImportError as e:
        print(f"  {e}")
        print("  Falling back to Forecast workbook...")
        holdings_df = load_holdings_from_forecast_workbook(FORECAST_WB)

print(f"\n{len(holdings_df)} positions, {holdings_df['CUSIP'].nunique()} unique CUSIPs")
holdings_df.head()

---
## Step 2: Enrich Holdings with Intex/BBG Data

Holdings from the clo library or Data Packet are missing Intex and Bloomberg fields.

| Source | Method | Fields |
|--------|--------|--------|
| **Bloomberg** | `blpapi` Python SDK (direct, no Excel) | BBG_NC_END, BBG_REINVEST_END, RESET_IDX, COLLAT_TYP |
| **IntexLINK** | Excel formulas (manual open or xlwings) | Intex Name, Deal Name, AAA Margin, Non-Call End, Reinvest End, Orig Deal Balance |

### Enrichment flow

1. **Bloomberg via blpapi** — fetches all 4 BBG fields in one bulk API call. No Excel needed.
2. **Write IntexLINK formula file** — creates `{DATE}_HoldingsEnrichment.xlsx` with Intex formulas + BBG values.
3. **Calculate Intex formulas** — two options:
   - `AUTO_CALC_INTEX = True`: xlwings opens in Excel, calculates, reads back (can be flaky with add-ins)
   - `AUTO_CALC_INTEX = False` (default): just writes the file — **open it in Excel yourself**, let formulas calculate, save, then **re-run this cell** to load it via `load_enriched_holdings()`

**If loading from the Forecast workbook** (HOLDINGS_SOURCE = "forecast_wb"), enrichment is skipped.

In [ ]:
# Pre-price deals: pass empty to skip (those CUSIPs get dropped by clean_holdings)
preprice = PrePriceDeals()

# ── ENRICHMENT ───────────────────────────────────────────────────────────
# Check if Intex/BBG columns are already present.
intex_bbg_cols = ["Intex Name", "Intex Deal Name", "AAA Margin",
                  "BBG_NC_END", "BBG_REINVEST_END", "RESET_IDX", "COLLAT_TYP"]
has_enrichment = all(c in holdings_df.columns and holdings_df[c].notna().any()
                     for c in intex_bbg_cols)

ENRICHMENT_FILE = OUTPUT_DIR / f"{TODAY}_HoldingsEnrichment.xlsx"

# Set to True to have xlwings auto-calculate Intex formulas in Excel.
# Set to False to just fetch BBG data + write the Intex formula file,
# then open it manually in Excel and use load_enriched_holdings() next run.
AUTO_CALC_INTEX = False

if has_enrichment:
    print("Holdings already have Intex/BBG data — skipping enrichment.")
elif ENRICHMENT_FILE.exists():
    # Load from a previously saved enrichment file (with calculated values).
    print(f"Loading previously enriched file: {ENRICHMENT_FILE.name}")
    holdings_df = load_enriched_holdings(ENRICHMENT_FILE, holdings_df)
else:
    # Fetch BBG data via blpapi + write Intex formula file.
    # If AUTO_CALC_INTEX=True, also opens in Excel to calculate Intex formulas.
    holdings_df = enrich_holdings(
        holdings_df, preprice, ENRICHMENT_FILE, auto_calc=AUTO_CALC_INTEX,
    )

# ── CLEAN ────────────────────────────────────────────────────────────────
# Cross-fill dates, drop CUSIPs with missing data.
holdings_df = clean_holdings(holdings_df)

# Convert to Tranche objects (one per unique CUSIP, face/par summed across positions)
tranches = export_tranches(holdings_df, preprice)

print(f"\n{len(tranches)} tranches ready for Intex")
print(f"{sum(1 for t in tranches if t.middle_market)} are middle market")

### Build scenarios from config

In [ ]:
scenarios = load_scenarios_from_config()

pd.DataFrame([
    {"#": s.number, "Name": s.name, "Settle": s.settle_date,
     "AAA Margin": s.initial_aaa_margin, "Shock": s.aaa_margin_shock,
     "Strike": s.initial_aaa_margin + s.aaa_margin_shock + CONFIG.call.refi_costs_bps}
    for s in scenarios
])

---
## Step 3: Clear Forecasted Factors

Start with a clean slate — factor call dates will be populated in Step 5
after running the preliminary cashflows through IntexCalc.

In [ ]:
factors = ForecastedFactors()
factors.clear()
print("Forecasted factors cleared.")

---
## Step 4: Generate Preliminary Intex Portfolio Upload

Creates a portfolio upload file with **COLLAT tranches** (one per deal per scenario).
The purpose is to get deal-level collateral cashflows from Intex so we can determine
when each deal's collateral has amortized enough to trigger a factor-based call.

Each row includes:
- Scenario parameters (settle date, spread shock)
- Call date based on refi economics (if the deal is in-the-money to be called)
- Prepayment assumption (CPR for the reinvestment period, then 0)
- Default/severity/recovery assumptions

In [ ]:
prelim_upload = generate_portfolio_upload(
    scenarios=scenarios,
    tranches=tranches,
    collateral_cashflows=True,  # COLLAT tranches for factor call extraction
)

prelim_path = OUTPUT_DIR / f"{TODAY}_PreliminaryPortfolioUpload.xlsx"
write_portfolio_upload(prelim_upload, prelim_path)

print(f"\nPreview of preliminary upload ({len(prelim_upload)} rows):")
prelim_upload.head()

### Step 4b: Run in IntexCalc (MANUAL)

**You must do this step outside Python:**

1. Open IntexCalc
2. Import the preliminary portfolio upload file: `output/{TODAY}_PreliminaryPortfolioUpload.xlsx`
3. Run cashflows
4. Export the cashflows report to Excel
5. Set the `PRELIMINARY_CASHFLOWS` variable at the top of this notebook to that exported file path
6. Continue to Step 5

---
## Step 5: Extract Factor Call Dates

From the preliminary COLLAT cashflows, find the first period where each deal's
balance drops below the **factor threshold** (default: 35% of original deal balance).
That date becomes the factor-based call date for that deal/scenario.

If you haven't run IntexCalc yet, this loads pre-existing factor data from the
Forecast workbook.

In [ ]:
if PRELIMINARY_CASHFLOWS:
    # Extract factor call dates from the Intex Cashflows export
    factors.import_from_cashflows_report(PRELIMINARY_CASHFLOWS, tranches)
    curve_df = load_from_cashflows_report(PRELIMINARY_CASHFLOWS)
else:
    # Load pre-existing factor data from the Forecast workbook
    factors.load_from_workbook(FORECAST_WB)
    curve_df = load_from_workbook(FORECAST_WB)

print(f"\nFactor call dates: {factors.count}")
print(f"Forward curve: {len(curve_df)} periods")

# Preview factor call dates
factors.to_dataframe().head(10)

---
## Step 6: Generate Final Intex Portfolio Upload

Now we generate the **final** portfolio upload with the **actual tranches** (not COLLAT).

The call date for each tranche in each scenario is the **earlier** of:
1. **Spread-based refi call**: Is it economic to refinance given the scenario's spread level?
2. **Factor call**: Has the deal's collateral amortized below 35%?

This is the key mechanism — **different spread scenarios produce different call dates**,
which produce different cashflow profiles.

In [ ]:
final_upload = generate_portfolio_upload(
    scenarios=scenarios,
    tranches=tranches,
    collateral_cashflows=False,  # Actual tranches with factor call dates
    forecasted_factors=factors,
)

final_path = OUTPUT_DIR / f"{TODAY}_FinalPortfolioUpload.xlsx"
write_portfolio_upload(final_upload, final_path)

# Show call decision summary
call_summary = final_upload["User Comment"].value_counts()
no_call = len(final_upload) - final_upload["User Comment"].notna().sum()
print(f"\nCall decisions across {len(final_upload)} rows:")
print(call_summary.to_string())
if no_call > 0:
    print(f"No call (Never):  {no_call}")

final_upload.head()

### Step 6b: Run in IntexCalc (MANUAL)

**You must do this step outside Python:**

1. Open IntexCalc
2. Import the final portfolio upload file: `output/{TODAY}_FinalPortfolioUpload.xlsx`
3. Run cashflows and analytics
4. Export the cashflows report to Excel
5. Set the `FINAL_CASHFLOWS` variable at the top of this notebook to that exported file path
6. Continue to Step 7

---
## Step 7: Summarize Cashflows & Allocate to Portfolios

Parse the final Intex cashflows export and create:
- **Per-scenario summary**: Interest, Principal, Balance for each tranche and portfolio total
- **Balance Summary**: Portfolio balance across all scenarios on one sheet
- **Sub-portfolio allocation**: Split cashflows by entity/PAM portfolio using par-weighted allocation

In [ ]:
if FINAL_CASHFLOWS:
    # Load and summarize cashflows
    reports = load_cashflows_reports(FINAL_CASHFLOWS)
    summary_path = OUTPUT_DIR / f"{TODAY}_CashflowSummary.xlsx"
    generate_cashflow_summary(reports, summary_path, scenarios)
    print(f"\nCashflow summary: {summary_path}")
else:
    print("No final cashflows file — set FINAL_CASHFLOWS at the top and re-run.")
    print("Skipping Steps 7 and 8.")

---
## Step 8: Sub-Portfolio Allocation

Allocate deal-level cashflows to sub-portfolios using par-weighted ownership:

```
weight = entity's Current Par for CUSIP / total Current Par for CUSIP
entity's cashflow = deal cashflow × weight
```

Define sub-portfolios by any combination of:
- **PAM Portfolio** numbers
- **Entity Names**
- **Client Level 3** values

In [ ]:
# ── DEFINE SUB-PORTFOLIOS ────────────────────────────────────────────────
# Option A: One sub-portfolio per Entity Name (automatic)
# sub_portfolios = one_per_entity(holdings_df)

# Option B: One sub-portfolio per PAM Portfolio (automatic)
# sub_portfolios = one_per_pam(holdings_df)

# Option C: Custom groupings (mix and match however you want)
sub_portfolios = [
    SubPortfolio("FHLB", pam_portfolios=[13091]),
    SubPortfolio("Fixed Annuity", pam_portfolios=[13015, 13035]),
    SubPortfolio("Pub Def Comp", pam_portfolios=[13013]),
    SubPortfolio("Registered ILA", entity_names=["NW Life Registered ILA"]),
    SubPortfolio("All Other", pam_portfolios=[
        415, 416, 13010, 13017, 13023, 13024, 13027,
        13036, 13043, 13049, 13090, 13094, 13330,
    ]),
    SubPortfolio("Total"),  # No filters = everything
]

# Compute ownership weights
weights = compute_weights(holdings_df, sub_portfolios)

# Summary
for sp_name in weights["SubPortfolio"].unique():
    sp_w = weights[weights["SubPortfolio"] == sp_name]
    print(f"  {sp_name}: {len(sp_w)} CUSIPs, ${sp_w['SubPortfolio Par'].sum()/1e9:.2f}B par")

In [ ]:
# Allocate cashflows to sub-portfolios
if FINAL_CASHFLOWS:
    allocated_path = OUTPUT_DIR / f"{TODAY}_AllocatedCashflows.xlsx"
    allocate_cashflows(summary_path, weights, allocated_path)
    print(f"\nAllocated cashflows: {allocated_path}")
else:
    print("Skipped — waiting for final cashflows.")

---
## Output Files

| File | Description |
|------|-------------|
| `{DATE}_PreliminaryPortfolioUpload.xlsx` | Upload to IntexCalc for COLLAT cashflows (Step 4) |
| `{DATE}_FinalPortfolioUpload.xlsx` | Upload to IntexCalc for tranche cashflows (Step 6) |
| `{DATE}_CashflowSummary.xlsx` | Per-scenario cashflow summary + Balance Summary (Step 7) |
| `{DATE}_AllocatedCashflows.xlsx` | Cashflows split by sub-portfolio (Step 8) |

In [ ]:
print("Output files:")
for f in sorted(OUTPUT_DIR.glob("*.xlsx")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

---
## Quick Reference: Config Parameters

All tunable values live in `forecast/config.py` and are accessible via `CONFIG`:

```python
from forecast.config import CONFIG, ScenarioDefinition
```

| Parameter | Default | Description |
|-----------|---------|-------------|
| `CONFIG.scenario.current_aaa_margin_bps` | 115 | Your view of current BSL AAA spreads (bps) |
| `CONFIG.scenario.settle_date` | today | Settlement date for all scenarios |
| `CONFIG.scenario.prepay_speed` | 15 | CPR assumption |
| `CONFIG.scenario.scenarios` | 8 scenarios | List of `ScenarioDefinition(name, shock_bps)` |
| `CONFIG.call.refi_costs_bps` | 10 | Refi friction cost (bps) |
| `CONFIG.call.middle_market_bsl_basis_bps` | 50 | MM AAA spread premium over BSL (bps) |
| `CONFIG.call.libor_sofr_basis_bps` | 26.161 | LIBOR-to-SOFR adjustment (bps) |
| `CONFIG.defaults.default_rate` | 5 | CDR (%) |
| `CONFIG.defaults.severity_pct` | 50 | Loss severity (%) |
| `CONFIG.defaults.recovery_lag_months` | 12 | Recovery lag (months) |
| `CONFIG.factor_call.factor_threshold` | 0.35 | Deal factor below which a call is triggered |